# Setup 🛠️

In [ ]:
# @title 🔑 Colab Secrets

from google.colab import userdata
import os

SECRET_KEYS = [
    "DEPLOY_TOKEN",
    "DEPLOY_USER",
    "DEPLOY_PASS",
    "MAPS_API_KEY",
    "REACT_APP_MAPS_API_KEY",
]

for key in SECRET_KEYS:
    try:
        os.environ[key] = userdata.get(key)
        print(f"✅ {key} exported.")
    except userdata.SecretNotFoundError:
        print(f"⚠️  {key} not set (add via 🔑 Secrets if needed).")

# Keep React / deploy env aliases in sync
if "REACT_APP_MAPS_API_KEY" not in os.environ and "MAPS_API_KEY" in os.environ:
    os.environ["REACT_APP_MAPS_API_KEY"] = os.environ["MAPS_API_KEY"]
elif "MAPS_API_KEY" not in os.environ and "REACT_APP_MAPS_API_KEY" in os.environ:
    os.environ["MAPS_API_KEY"] = os.environ["REACT_APP_MAPS_API_KEY"]


In [ ]:
# @title 🛠️ Initialize RAM Drive & Base Tools
import os
import time

# 1. Setup RAM Drive (default 6GB; override with RAM_DRIVE_MB env var)
RAM_MB = int(os.environ.get('RAM_DRIVE_MB', '6144'))
if not os.path.exists("/content/build_space"):
  !sudo mkdir /content/build_space
  !sudo mount -t ramfs -o size={RAM_MB}M ramfs /content/build_space
  !sudo chmod 0777 -R /content/build_space
  print(f"✅ RAM Drive mounted at /content/build_space ({RAM_MB} MB)")

# 2. Install Generic Build Tools
print("⬇️ Installing Base Build Tools (CMake, Ninja, Zip)...")
!sudo apt-get update -qq
!sudo apt-get install -y build-essential cmake ninja-build unzip git curl wget python3-pip
!sudo apt-get install -y make libsdl2-dev fontconfig
!python3 -m pip install --upgrade pip
!pip install paramiko
%cd /content/build_space

In [ ]:
# @title
%%shell
# Install Node 22.x (Current LTS)
curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
sudo apt-get install -y nodejs
sudo apt-get install -y aptitude

# Install global tools (Vite, Typescript, etc.)
sudo npm install -g npm@latest
sudo npm install -g svn uuid google-closure-compiler commander terser wasm-opt advzip-bin roadroller
sudo aptitude install binaryen uglifyjs.terser -y
sudo npm install -g typescript vite yarn http-server assemblyscript

echo "✅ Node Version:"
node -v
echo "✅ NPM Version:"
npm -v


In [ ]:
# @title
%%shell
#!/bin/bash
set -e

RED=$'\e[0;31m'
GREEN=$'\e[0;32m'
YELLOW=$'\e[0;33m'
NC=$'\e[0m' # No Color
PYTHON_EXECUTABLE="${PYTHON_EXECUTABLE:=}"
INSTALL_PY_URL="${INSTALL_PY_URL:=}"
INSTALL_PY_PATH="${INSTALL_PY_PATH:=}"

if ! command -v git &>/dev/null; then
    echo "${RED}Please install git${NC}"
    exit 1
fi

main() {
    if [ "$PYTHON_EXECUTABLE" = "" ]; then
        if ! command -v which &>/dev/null; then
            echo "${RED}Please install python or provide python path:"
            echo "PYTHON_EXECUTABLE=<path> install.sh"
            echo "Please install which${NC}"
            exit 1
        fi
        if command -v python3 &>/dev/null; then
            PYTHON_EXECUTABLE="$(which python3)"
        elif command -v python2 &>/dev/null; then
            PYTHON_EXECUTABLE="$(which python2)"
        elif command -v python &>/dev/null; then
            PYTHON_EXECUTABLE="$(which python)"
        else
            echo "${RED}Please install python or provide python path:"
            echo "PYTHON_EXECUTABLE=<path> install.sh${NC}"
            exit 1
        fi
        echo "${YELLOW}Using Python: $PYTHON_EXECUTABLE ${NC}"
    else
        echo "${GREEN}Using Python: $PYTHON_EXECUTABLE ${NC}"
    fi

    if [ "$INSTALL_PY_PATH" != "" ]; then
        # If a local installer script is given, then use it directly
        "$PYTHON_EXECUTABLE" "$INSTALL_PY_PATH" "$@"
        exit 0
    fi

    if [ "$INSTALL_PY_URL" = "" ]; then
        INSTALL_PY_URL="https://raw.githubusercontent.com/WasmEdge/WasmEdge/master/utils/install.py"
    fi

    if command -v curl &>/dev/null; then
        if curl --output /dev/null --silent --head --fail "$INSTALL_PY_URL"; then
            curl -sSf "$INSTALL_PY_URL" | "$PYTHON_EXECUTABLE" - "$@"
        else
            echo "${RED}$INSTALL_PY_URL not reachable${NC}"
        fi

    elif command -v wget &>/dev/null; then
        if wget -q --method=HEAD "$INSTALL_PY_URL"; then
            wget -qO- "$INSTALL_PY_URL" | "$PYTHON_EXECUTABLE" - "$@"
        else
            echo "${RED}$INSTALL_PY_URL not reachable${NC}"
        fi
    else
        echo "${RED}curl or wget could not be found${NC}"
        exit 1
    fi

}

main "$@"

In [ ]:
%%shell
# @title
curl -sSf https://raw.githubusercontent.com/WasmEdge/WasmEdge/master/utils/install.sh | bash
source $HOME/.wasmedge/env
mkdir /content/build_space/wasmedge
cp /root/.wasmedge/include/wasmedge/* -r /content/build_space/wasmedge/


In [ ]:
# @title 🛠️ Install Emscripten (EMSDK)
%%shell
cd /content/build_space

if [ ! -d "emsdk" ]; then
  echo "⬇️ Cloning EMSDK..."
  git clone https://github.com/emscripten-core/emsdk.git
fi

cd emsdk
git pull

./emsdk install tot
./emsdk activate tot

echo "✅ Emscripten Installed. Remember to source environment in your build cells:"
echo "source /content/build_space/emsdk/emsdk_env.sh"

In [ ]:
# @title 🛠️ Install Rust & Wasm-Pack
%%shell
# Install Rust
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
source "$HOME/.cargo/env"
# Install wasm-pack via cargo to avoid TTY issues
cargo install wasm-pack

# Add requested targets
rustup target add wasm32-unknown-unknown
rustup target add i686-unknown-linux-gnu

echo "✅ Rust and wasm-pack Installed."
rustc --version
wasm-pack --version

In [ ]:
# @title 🛠️ APT Update
!sudo apt update
!sudo apt upgrade -y
!sudo apt install libhwloc-dev libuv1-dev

# GLM + assimp

In [ ]:
# @title 🛠️ Install GLM
!sudo apt-get install -y libglew-dev zzip-zlib-config

# GLM is header-only; clone for include path
!rm -rf /content/build_space/glm
!git clone --depth 1 https://github.com/g-truc/glm.git /content/build_space/glm
print("✅ GLM headers available at /content/build_space/glm")


In [ ]:
# @title 🛠️ Install assimp
!sudo apt-get install -y zzip-zlib-config
!source /content/build_space/emsdk/emsdk_env.sh
!rm -rf /content/build_space/assimp
!git clone --depth 1 https://github.com/assimp/assimp.git /content/build_space/assimp
!mkdir -p /content/build_space/assimp/build
!cd /content/build_space/assimp/build && emcmake cmake .. \
        -DCMAKE_BUILD_TYPE=Release \
        -DASSIMP_BUILD_TESTS=OFF \
        -DASSIMP_BUILD_ASSIMP_TOOLS=OFF \
        -DASSIMP_BUILD_SAMPLES=OFF \
        -DASSIMP_NO_EXPORT=ON \
        -DASSIMP_BUILD_ZLIB=ON

!cd /content/build_space/assimp/build && emmake make -j$(nproc)


# BUILD 🔨

In [ ]:
# @title 🔧 Build & Deploy Helpers
import os
import subprocess

BUILD_SPACE = "/content/build_space"
EMS_SOURCE = f"{BUILD_SPACE}/emsdk/emsdk_env.sh"


def _run(cmd, cwd=None, env=None, check=True):
    merged = os.environ.copy()
    if env:
        merged.update(env)
    print(f"$ {cmd}" + (f"  (cwd={cwd})" if cwd else ""))
    result = subprocess.run(cmd, shell=True, cwd=cwd, env=merged)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result


def clone_or_pull(project_name, branch="main", recursive=False, git_repo=None):
    target = f"{BUILD_SPACE}/{project_name}"
    repo = git_repo or f"https://github.com/ford442/{project_name}.git"
    recursive_flag = "--recursive " if recursive else ""
    if not os.path.exists(target):
        print(f"⬇️ Cloning {repo}...")
        _run(f"git clone {recursive_flag}{repo} {target}")
    else:
        print(f"🔄 Pulling {project_name}...")
        _run("git pull", cwd=target)
    _run(f"git checkout {branch}", cwd=target, check=False)
    _run("git pull", cwd=target, check=False)
    return target


def run_npm_build(project_name, use_emsdk=False, cwd=None):
    cwd = cwd or f"{BUILD_SPACE}/{project_name}"
    if use_emsdk:
        _run(
            f"bash -c 'source {EMS_SOURCE} && npm install && npm run build'",
            cwd=cwd,
        )
    else:
        _run("npm install", cwd=cwd)
        _run("npm run build", cwd=cwd)


def patch_html_to_relative(html_path, extra_replacements=None):
    if not os.path.exists(html_path):
        raise FileNotFoundError(f"HTML not found: {html_path}")
    with open(html_path, "r", encoding="utf-8") as f:
        content = f.read()
    replacements = [
        ('src="/', 'src="./'),
        ('href="/', 'href="./'),
        ("src='/", "src='./"),
        ("href='/", "href='./"),
    ]
    if extra_replacements:
        replacements = list(extra_replacements) + replacements
    for old, new in replacements:
        content = content.replace(old, new)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"✅ Patched paths in {html_path}")


def finalize_deploy(
    project_name,
    html_subpath="dist/index.html",
    deploy_script="deploy.py",
    work_subdir=None,
    extra_env=None,
    extra_replacements=None,
    skip_deploy=False,
    skip_patch=False,
):
    base = f"{BUILD_SPACE}/{project_name}"
    if work_subdir:
        base = f"{base}/{work_subdir}"
    html_path = (
        html_subpath
        if html_subpath.startswith("/")
        else f"{base}/{html_subpath}"
    )
    if not os.path.exists(html_path):
        raise FileNotFoundError(
            f"Build output missing: {html_path}. Did the build succeed?"
        )
    if not skip_patch:
        patch_html_to_relative(html_path, extra_replacements)
    out_dir = os.path.dirname(html_path)
    ink_path = f"{out_dir}/1ink.1ink"
    _run(f"iconv -f UTF-8 -t UTF-16 {html_path} -o {ink_path}")
    if not skip_deploy:
        deploy_cwd = base
        if not os.path.exists(os.path.join(deploy_cwd, deploy_script)):
            deploy_cwd = f"{BUILD_SPACE}/{project_name}"
        _run(f"python3 {deploy_script}", cwd=deploy_cwd, env=extra_env)
    print(f"✅ Deploy complete for {project_name}")


def sftp_upload(local_path, remote_path, host=None, username=None, password=None, port=22):
    import paramiko

    host = host or os.environ.get("SFTP_HOST", "1ink.us")
    username = username or os.environ.get("DEPLOY_USER", os.environ.get("SFTP_USER", ""))
    password = password or os.environ.get("DEPLOY_PASS", os.environ.get("SFTP_PASSWORD", ""))
    if not username or not password:
        raise RuntimeError(
            "SFTP credentials missing. Set DEPLOY_USER and DEPLOY_PASS in Colab secrets."
        )
    transport = paramiko.Transport((host, port))
    transport.connect(username=username, password=password)
    sftp = paramiko.SFTPClient.from_transport(transport)
    sftp.put(local_path, remote_path)
    sftp.close()
    transport.close()
    print(f"✅ Uploaded {local_path} → {remote_path}")


In [ ]:
# @title 🔨 go.1ink.us
import os

PROJECT_NAME = "go.1ink.us"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 1inkulous
import os

PROJECT_NAME = "1inkulous"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Plant Growth Sim
import os

PROJECT_NAME = "plant-growth-sim"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 rebirth website
import os

PROJECT_NAME = "rebirth_website"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Wind Manager
import os

PROJECT_NAME = "wind_manager"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 XMRig
import os
PROJECT_NAME = "xmrig_wasm" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main
#!git checkout copilot/add-css-formatting-3d-panels
!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    #!bash /content/build_space/xmrig_wasm/scripts/build_deps.sh

    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run configure && npm run build
else:
    # Standard NPM Build
    !bash /content/build_space/xmrig_wasm/scripts/build_deps.sh
    !npm install
    !rm -rf build_wasm/CMakeCache.txt build_wasm/CMakeFiles # Clean stale cmake cache
    !npm run configure
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.
!cp /content/build_space/{PROJECT_NAME}/wasm_test.html /content/build_space/{PROJECT_NAME}/dist/index.html

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

!cd /content/build_space/{PROJECT_NAME}/dist && iconv -f UTF-8 -t UTF-16 ./index.html -o ./1ink.1ink

!iconv -f UTF-8 -t UTF-16 /content/build_space/xmrig_wasm/dist/xmrig.js -o /content/build_space/xmrig_wasm/dist/xmrig.1ijs
!iconv -f UTF-8 -t UTF-32 /content/build_space/xmrig_wasm/dist/xmrig.js -o /content/build_space/xmrig_wasm/dist/xmrig.3ijs
# PROJECT_NAME is defined in a previous cell and inherited here.
!python3 deploy.py

In [ ]:
# @title 🔨 Weeks on Fire
import os

PROJECT_NAME = "weeks_on_fire"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "out/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Yoga
import os

PROJECT_NAME = "yoga_studio"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "out/index.html"
DEPLOY_SCRIPT = "deploy_old.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 scheduler
import os

PROJECT_NAME = "scheduler"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Benching Machine
import os

PROJECT_NAME = "benching_machine" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = True

target_dir = f"/content/build_space/{PROJECT_NAME}"

if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main

!git pull

if USE_EMSDK:
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    print("🚀 NPM Install...")
    !npm install

    print("🚀 Starting C++ Build...")
    !npm run build:physics
    print("🚀 Starting OpenMP Build...")
    !npm run build:omp
    print("🚀 Starting NPM Build...")
    !npm run build
else:
    print("🚀 NPM Install...")
    !npm install

    print("🚀 Starting C++ Build...")
    !npm run build:physics
    print("🚀 Starting OpenMP Build...")
    !npm run build:omp
    print("🚀 Starting NPM Build...")
    !npm run build

print("✅ Build Complete.")

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/build/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./build/index.html -o ./build/1ink.1ink
!python3 deploy.py

In [ ]:
# @title 🔨 UTF-16 Bench Marker
import os

PROJECT_NAME = "benching_machine" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = True

target_dir = f"/content/build_space/{PROJECT_NAME}"

if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}/utf_benchmark

!git checkout main

!git pull

if USE_EMSDK:
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    print("🚀 NPM Install...")
    !npm install

    print("🚀 Starting C++ Build...")
    !npm run build:physics
    print("🚀 Starting OpenMP Build...")
    !npm run build:omp
    print("🚀 Starting NPM Build...")
    !npm run build
else:
    print("🚀 NPM Install...")
    !npm install

    print("🚀 Starting C++ Build...")
    !npm run build:physics
    print("🚀 Starting OpenMP Build...")
    !npm run build:omp
    print("🚀 Starting NPM Build...")
    !npm run build

print("✅ Build Complete.")

target_dir = f"/content/build_space/{PROJECT_NAME}/utf_benchmark"
html_file_path = f'{target_dir}/utf_benchmark/dist/utf8-control/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="dist/utf8-control/index.html", deploy_script="deploy.py", work_subdir="utf_benchmark", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 Dice Roller
import os

PROJECT_NAME = "webgl-diceroller"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Brain Visualizer
import os

PROJECT_NAME = "brain_viz"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Clip Stacker
import os

PROJECT_NAME = "clip_stacker"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy_old.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Power Gen
import os

PROJECT_NAME = "power_gen"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Prophecy
import os

# --- CONFIGURATION ---
PROJECT_NAME = "korg_prophecy_emu" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main

!git pull
# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/build/web/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="build/web/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 Chromashift
import os

# --- CONFIGURATION ---
PROJECT_NAME = "chromashift" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main
#!git checkout 45ee7bcf5394c187b91d37e68a0c34cded6666fb
#!git checkout edabd9e5c1de0b3e5dce11f7ca4902e4144f940b

!git pull
# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && cd /content/build_space/chromashift/cpp/ && emmake make -j && npm install && npm run build
else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py  # DEPLOY_USER / DEPLOY_PASS from Colab secrets

In [ ]:
# @title 🔨 Harborglow
import os

PROJECT_NAME = "harborglow"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Zephyr
import os

PROJECT_NAME = "grok_zephyr"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨Marbles
import os

# --- CONFIGURATION ---
PROJECT_NAME = "marbles" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main

!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
!cp /content/build_space/marbles/baked_color.filmat /content/build_space/marbles/dist/baked_color.filmat
!cp /content/build_space/marbles/node_modules/filament/filament.wasm /content/build_space/marbles/dist/filament.wasm
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py
#!python3 deploy_old.py

In [ ]:
# @title 🔨Web Sequencer
import os
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "web_sequencer"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"

target_dir = f"/content/build_space/{PROJECT_NAME}"

# 1. Setup Environment (Rust + Wasm-Pack)
print("⬇️ Checking Rust & Wasm-Pack environment...")

# Ensure ~/.cargo/bin is in PATH for this session (Python side)
cargo_bin_dir = os.path.expanduser("~/.cargo/bin")
if cargo_bin_dir not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + cargo_bin_dir

# Check wasm-pack
if shutil.which("wasm-pack"):
    print(f"✅ wasm-pack is available.")
else:
    print("❌ wasm-pack not found. Please run the 'Install Rust & Wasm-Pack' setup cell first.")

# 2. Clone / Update Repo
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir} --recursive
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
!git checkout main
!git pull
!git submodule update --init --recursive

# Build WASM binary
print("🚀 Building jc303 WASM binary...")
!source /content/build_space/emsdk/emsdk_env.sh && bash tools/build_jc303_omp.sh release single

# 3. Prepare Dependencies & Assets
print("📦 Installing Dependencies & Preparing Assets...")
!npm install

# Fix for rubberband location - ensure it exists in root if build script looks there
if not os.path.exists("rubberband"):
    print("⬇️ Cloning rubberband to project root...")
    !git clone https://github.com/breakfastquay/rubberband.git rubberband
else:
    print("🔄 rubberband (root) exists. Pulling...")
    !cd rubberband && git pull

# Also ensure it exists in emscripten/rubberband as per original script
if not os.path.exists("emscripten/rubberband"):
     print("⬇️ Cloning rubberband to emscripten/rubberband...")
     !mkdir -p emscripten
     !git clone https://github.com/breakfastquay/rubberband.git emscripten/rubberband
else:
     print("🔄 emscripten/rubberband exists. Pulling...")
     !cd emscripten/rubberband && git pull

# Copy onnxruntime-web assets to public/assets BEFORE build
public_assets_dest = os.path.join(target_dir, "public", "assets")
if not os.path.exists(public_assets_dest):
    os.makedirs(public_assets_dest, exist_ok=True)
    print("Copying onnxruntime-web assets...")
    !cp -r node_modules/onnxruntime-web/dist/* {public_assets_dest}/ 2>/dev/null || true

# 4. Build
print("🚀 Starting Build...")
# Ensure PATH includes everything needed, including cargo bin, wasmedge bin, and npm bin locations
!export PATH=$HOME/.cargo/bin:$HOME/.wasmedge/bin:$PATH && npm run build

# 5. Post-Build Actions
dist_dir = os.path.join(target_dir, "dist")
if os.path.exists(dist_dir):
    print("✅ Build Complete.")

    # Copy .wav files to dist/assets
    assets_dest = os.path.join(dist_dir, "assets")
    os.makedirs(assets_dest, exist_ok=True)

    public_wavs = os.path.join(target_dir, "public")
    if os.path.exists(public_wavs):
        print(f"Copying .wav files from {public_wavs} to {assets_dest}...")
        !cp {public_wavs}/*.wav {assets_dest}/ 2>/dev/null || true

    # Patch Index HTML
    html_file_path = os.path.join(dist_dir, "index.html")
    if os.path.exists(html_file_path):
        print(f"Patching {html_file_path}...")
        try:
            with open(html_file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            content = content.replace('src="/', 'src="./')
            content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
            with open(html_file_path, 'w', encoding='utf-8') as file:
                file.write(content)
            print("✅ Successfully updated paths in index.html to be relative.")

            # Create 1ink file
            !iconv -f UTF-8 -t UTF-16 {html_file_path} -o {dist_dir}/1ink.1ink

            # Deploy
            !python3 deploy.py
            #!python3 deploy_old.py
        except Exception as e:
             print(f"❌ An error occurred during patching: {e}")
    else:
        print(f"❌ index.html not found in {dist_dir}")

else:
    print("❌ Build failed (dist directory not found).")

In [ ]:
# @title 🔨 weather_clock
import os

PROJECT_NAME = "weather_clock"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨Tetris WebGPU
import os

# --- CONFIGURATION ---
PROJECT_NAME = "Tetris_WebGPU" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

#good
#!git checkout 185d33c4ad68a531dfb45687c781f772a933a046

#glass
#!git checkout 048b073f312756607e832d2c77d0adf4cb879246
#!git checkout 8a0f6a979edc85556fc67afebfbb94147becac08

#test
!git checkout main

#!git checkout restore-0126
#!git checkout master
#!git checkout texture
#!git checkout main

!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

!cp /content/build_space/Tetris_WebGPU/build/release.wasm /content/build_space/Tetris_WebGPU/dist/release.wasm
# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py

In [ ]:
# @title 🔨 candy_world
import os

PROJECT_NAME = "candy_world"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = True
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨MOD Player
import os
import re
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "mod-player"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
!git checkout main
!git pull

# --- PRE-BUILD FIXES ---
print("🔧 Applying pre-build fixes...")

# Fix 1: Unused import in useLibOpenMPT.ts
hook_path = os.path.join(target_dir, "hooks/useLibOpenMPT.ts")
if os.path.exists(hook_path):
    try:
        with open(hook_path, "r") as f:
            c = f.read()
        # Regex to find the unused import line
        # Matches: import { withBase } from '../src/lib/paths';
        pattern = r"import\s+\{\s*withBase\s*\}\s+from\s+['\"]\.\./src/lib/paths['\"];?"
        if re.search(pattern, c):
            c = re.sub(pattern, "// Unused import removed by build script", c)
            with open(hook_path, "w") as f:
                f.write(c)
            print("  ✅ Removed unused 'withBase' import in useLibOpenMPT.ts")
    except Exception as e:
        print(f"  ⚠️ Failed to patch useLibOpenMPT.ts: {e}")


# --- BUILD COMMANDS ---
print("🚀 Starting Build...")
!export VITE_APP_BASE_PATH=/xm-player/
# Ensure clean slate for build
if os.path.exists("dist"):
    shutil.rmtree("dist")

# Use standard ! magic for build to ensure environment is correct
if USE_EMSDK:
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    !npm install
    !npm run build  #:xm-player:verify

if not os.path.exists("dist/index.html"):
    print("❌ Build failed! dist/index.html not found.")
else:
    print("✅ Build Verification Passed.")

    # --- CRITICAL FIX: Ensure worklet files are in dist ---
    worklet_src = f'{target_dir}/public/worklets'
    worklet_dist = f'{target_dir}/dist/worklets'

    if os.path.exists(worklet_src):
        os.makedirs(worklet_dist, exist_ok=True)
        for f in os.listdir(worklet_src):
            src_file = os.path.join(worklet_src, f)
            dst_file = os.path.join(worklet_dist, f)
            if os.path.isfile(src_file):
                shutil.copy2(src_file, dst_file)
                print(f"📄 Copied worklet: {f}")

        # Create openmpt-native.js as copy of worklet (prevents 404)
        native_js = f'{worklet_dist}/openmpt-native.js'
        worklet_js = f'{worklet_dist}/openmpt-worklet.js'
        if os.path.exists(worklet_js) and not os.path.exists(native_js):
            shutil.copy(worklet_js, native_js)
            print("📄 Created openmpt-native.js (copy of worklet)")

    # --- PATCH INDEX.HTML ---
    html_file_path = f'{target_dir}/dist/index.html'
    print(f"\n🔧 Patching {html_file_path}...")

    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # 1. Generic Relative Path Fix (Handle both double and single quotes)
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        print("✅ Converted absolute paths to relative.")

        # 2. Fix CSS filename for 1ink (rename file + update HTML)
        # We look for the CSS file in the HTML (now relative)
        # Regex matches: href="./assets/somename.css" with either ' or " quotes
        # Using triple-quoted string to safely handle all quote types
        css_regex = r"""href=["'](\./assets/[^"']+\.css)["']"""
        css_match = re.search(css_regex, content)

        if css_match:
            old_css_rel_path = css_match.group(1) # e.g. ./assets/index-xxxxx.css
            old_css_name = os.path.basename(old_css_rel_path) # e.g. index-xxxxx.css

            src_css = os.path.join(target_dir, 'dist', 'assets', old_css_name)
            dst_css = os.path.join(target_dir, 'dist', 'assets', 'modplayer.1iss')

            # Check if source exists
            if os.path.exists(src_css):
                # Avoid renaming if source and dest are the same (idempotency)
                if os.path.abspath(src_css) != os.path.abspath(dst_css):
                    shutil.move(src_css, dst_css)
                    print(f"✅ Renamed {old_css_name} -> modplayer.1iss")
                else:
                     print(f"ℹ️ CSS file already renamed to modplayer.1iss")

                # Update HTML to point to new name
                content = content.replace(old_css_rel_path, './assets/modplayer.1iss')
                print("✅ Updated HTML to link to modplayer.1iss")
            elif os.path.exists(dst_css):
                 print("ℹ️ modplayer.1iss already exists. Updating HTML to ensure link is correct.")
                 content = content.replace(old_css_rel_path, './assets/modplayer.1iss')
            else:
                print(f"⚠️ Could not find CSS file {src_css} to rename!")
        else:
            print("⚠️ No CSS link found in index.html to patch.")

        # 3. Add base tag to help with relative path resolution
        if '<base' not in content and '<head>' in content:
            content = content.replace('<head>', '<head>\n  <base href="./">')
            print("✅ Added <base href=\"./\">")

        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully saved patched index.html")

    except Exception as e:
        print(f"❌ Error patching: {e}")

    # --- DEPLOY ---
    print("\n📤 Deploying...")
    if os.path.exists("deploy.py"):
        !export VITE_APP_BASE_PATH=/xm-player/
        !python3 deploy.py
    else:
        print("⚠️ deploy.py not found, skipping deployment.")
    print("🎉 Done!")

In [ ]:
# @title 🔨 Pachinball
import os

# --- CONFIGURATION ---
PROJECT_NAME = "pachinball" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

#!git checkout 216673b05dccc7cf2678debae04f537ac450778f

!git checkout main

!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build_colab
else:
    # Standard NPM Build
    !npm install
    !npm run build_colab

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!rm /content/build_space/{PROJECT_NAME}/dist/backbox/.gitkeep
!python3 deploy.py

In [ ]:
# @title 🔨 image_video_effects
import os

# --- CONFIGURATION ---
PROJECT_NAME = "image_video_effects" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main
#!git checkout main-dev2
#!git checkout stack-shaders-13277186508483700298
#!git checkout copilot/implement-advanced-shaders
#!git checkout fix-wgsl-compute-derivatives
#!git checkout feature/rain-shader
!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

else:
    # Standard NPM Build

    !npm install
    !npm install hls.js --save
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/build/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
!cp /content/build_space/image_video_effects/src/shader_coordinates.json /content/build_space/image_video_effects/build/

# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./build/index.html -o ./build/1ink.1ink
!python3 scripts/deploy_app_only.py

In [ ]:
# @title 🔨 Project-M
import os

PROJECT_ROOT = "/content/build_space/projectm"
GIT_REPO = "https://github.com/ford442/Project-M.git"
BRANCH = "main"  # @param {type:"string"}

# Optional: set before running build cell
# os.environ["RUN_OPTIMIZE"] = "1"       # run optimize.sh after bundle staging
# os.environ["BUILD_JOBS"] = "8"
# os.environ["ENABLE_WASM_TRANSITIONS"] = "ON"

if not os.path.exists(PROJECT_ROOT):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone --recursive {GIT_REPO} {PROJECT_ROOT}
else:
    print("🔄 Pulling latest...")
    %cd {PROJECT_ROOT}
    !git pull --recurse-submodules

%cd {PROJECT_ROOT}
!git checkout {BRANCH}
!git submodule update --init --recursive
!git pull --recurse-submodules

print("🚀 Building via scripts/colab_build.sh (aligned with repo CI)...")
!chmod +x scripts/colab_build.sh scripts/colab_deploy.sh
!{PROJECT_ROOT}/scripts/colab_build.sh

print("📦 Deploying via scripts/colab_deploy.sh...")
!{PROJECT_ROOT}/scripts/colab_deploy.sh

print("✅ Project-M build and deploy complete.")


In [ ]:
# @title 🔨 ui_componants
import os
import glob

# --- CONFIGURATION ---
PROJECT_NAME = "ui_componants" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main

!git pull


# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build

print("✅ Build Complete.")

# --- PATCHING ---
print("🩹 Patching HTML paths...")
dist_dir = f"{target_dir}/dist"

def patch_file(file_path, replacement_prefix):
    if not os.path.exists(file_path):
        return

    try:
        print(f"  Processing {os.path.basename(file_path)}...")
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # Replace absolute paths with relative ones based on depth
        # We replace src="/ and href="/ with the relative prefix
        original_len = len(content)
        content = content.replace('src="/', f'src="{replacement_prefix}')
        content = content.replace('href="/', f'href="{replacement_prefix}')

        if len(content) != original_len:
            with open(file_path, 'w', encoding='utf-8') as file:
                file.write(content)
            print(f"    ✅ Patched links with prefix '{replacement_prefix}'")
        else:
            print("    ℹ️ No absolute links found.")

    except Exception as e:
        print(f"    ❌ Error patching {file_path}: {e}")

# 1. Patch root index.html (Prefix: ./)
patch_file(f"{dist_dir}/index.html", "./")

!cp /content/build_space/{PROJECT_NAME}/public/* /content/build_space/{PROJECT_NAME}/dist/
# 2. Patch files in pages/ subdirectory (Prefix: ../)
pages_dir = f"{dist_dir}/pages"
if os.path.exists(pages_dir):
    for filename in os.listdir(pages_dir):
        if filename.endswith(".html"):
            patch_file(os.path.join(pages_dir, filename), "../")

!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink

# --- DEPLOY ---
# PROJECT_NAME is defined in a previous cell and inherited here.
print("🚀 Deploying...")
!cd {target_dir} && python3 deploy.py

In [ ]:
# @title 🔨 Watershed
import os

# --- CONFIGURATION ---
PROJECT_NAME = "Watershed" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout main

!git pull


# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build_colab

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/build/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="build/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 Dog Dash
import os

PROJECT_NAME = "dog_dash"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 BeSpoke Synth
import os
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "BespokeSynth_WASM" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"
base_dir = "/content/build_space"

# Ensure we are in a safe directory
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
%cd {base_dir}

# 1. CLEANUP: Check for corrupt directory (exists but no .git)
if os.path.exists(target_dir) and not os.path.exists(os.path.join(target_dir, ".git")):
    print(f"🗑️ Found corrupt directory at {target_dir}. Removing...")
    !rm -rf {target_dir}

# 2. CLONE OR UPDATE
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir} --recursive
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

# 3. BUILD PREPARATION
if os.path.exists(target_dir):
    %cd {target_dir}
    !git checkout main
    !git pull

    # Fix for nanovg clone
    nanovg_dir = f"{target_dir}/libs/nanovg"
    if os.path.exists(nanovg_dir):
        print(f"🗑️ Removing existing NanoVG at {nanovg_dir} to ensure clean clone...")
        !rm -rf {nanovg_dir}

    print("⬇️ Cloning NanoVG...")
    !cd {target_dir}/libs && git clone https://github.com/memononen/nanovg.git

    # Mock missing 'leathers' library
    !mkdir -p {target_dir}/libs/leathers
    !touch {target_dir}/libs/leathers/push
    !cd {target_dir}/wasm && rm -rf CMakeFiles CMakeCache.txt

    !cd {target_dir}/libs && mkdir -p leathers
    !cd {target_dir}/libs && touch leathers/unused-variable

    # Fix NanoVG include structure
    !mkdir -p {target_dir}/libs/nanovg_fix/nanovg
    !cp {target_dir}/libs/nanovg/src/*.h {target_dir}/libs/nanovg_fix/nanovg/

    # WebGPU headers setup
    webgpu_include_dir = f"{target_dir}/wasm/include/webgpu"
    if not os.path.exists(webgpu_include_dir):
        !git clone https://github.com/eliemichel/WebGPU-Cpp.git {webgpu_include_dir}

    print("⚙️ Generating WebGPU headers...")
    !cd {webgpu_include_dir} && python generate.py -u dawn/webgpu.h -t dawn/webgpu.template.hpp -o dawn/webgpu.hpp --use-init-macros
    !cd {webgpu_include_dir} && python generate.py -u emscripten/webgpu.h -t emscripten/webgpu.template.hpp -o emscripten/webgpu.hpp --use-init-macros

    # Fix generated header location
    src_header = f'{webgpu_include_dir}/emscripten/webgpu.h'
    dest_header = f'{webgpu_include_dir}/webgpu.h'

    if os.path.exists(src_header):
        print(f"✅ Found generated header at {src_header}")
        shutil.copy2(src_header, dest_header)
        print(f"✅ Copied to {dest_header}")
    else:
        print(f"❌ Could not find source header at {src_header}")

    # --- BUILD COMMANDS ---
    print("🚀 Starting Build...")
    # Ensure we are in the correct directory before building
    !cd {target_dir} && npm install && npm run build

    print("✅ Build Complete.")

    # --- PATCHING & DEPLOY ---
    html_file_path = f'{target_dir}/dist/index.html'
    print(f"Attempting to patch {html_file_path}...")
    if not os.path.exists(html_file_path):
        print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
    else:
        try:
            with open(html_file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            content = content.replace('src="/', 'src="./')
            content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
            with open(html_file_path, 'w', encoding='utf-8') as file:
                file.write(content)
            print("✅ Successfully updated paths in index.html to be relative.")
        except Exception as e:
            print(f"❌ An error occurred during patching: {e}")

    # Create 1ink file and cleanup
    !cd {target_dir} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
    !rm -rf {target_dir}/dist/resource
    !rm -rf {target_dir}/wasm/dist/resource
    !rm -rf {target_dir}/dist/wasm/resource

    # Deploy
    !cd {target_dir} && python3 deploy.py
else:
    print(f"❌ Critical Error: Target directory {target_dir} could not be created or accessed.")

In [ ]:
# @title 🔨 cave_crystals
import os

PROJECT_NAME = "cave_crystals"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 SolarSystem 3d
import os
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "SolarSystem-3D-WASM" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True to rebuild C++ to WASM
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir} --recursive
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}/web
!git pull

!git checkout main

!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    %cd {target_dir}
    !source /content/build_space/emsdk/emsdk_env.sh && ./build-web.sh
    %cd {target_dir}/web
    !npm install
    !npm run build
else:
    %cd {target_dir}/web
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/web/dist/index.html' # Note: Assuming it builds to web/dist
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # Fix the double path issue
        content = content.replace('src="/solar-system/', 'src="./')
        content = content.replace('href="/solar-system/', 'href="./')
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")

        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME}/web && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink

!python3 deploy.py


In [ ]:
# @title 🔨 Cloud Notes
import os

PROJECT_NAME = "cloud_notes"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Rain Edit
import os

PROJECT_NAME = "rain_edit"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 webgpu_streetview - Clean Build (Fixed)
import os
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "webgpu_streetview"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
# ---------------------
# MAPS_API_KEY / REACT_APP_MAPS_API_KEY loaded from Colab secrets (run 🔑 cell first)
target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone or pull latest
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir} # -b 2a94c9736a74dea7d2106d991c0ea71ad2512f35
else:
    print(f"🔄 Repository exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
!git checkout main
#!git checkout 2a94c9736a74dea7d2106d991c0ea71ad2512f35

##!git checkout 6e59aa3cf1c685294271e6ac84f7159190d0620d
!git pull

# --- CLEAN BUILD ---
print("\n🧹 Cleaning previous build...")
build_path = f"{target_dir}/build"
if os.path.exists(build_path):
    shutil.rmtree(build_path)
    print("   → Old build folder deleted")

# --- BUILD ---
print("\n🚀 Starting Build...")

if USE_EMSDK:
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Write .env.local (using your current key)
    !echo "REACT_APP_MAPS_API_KEY=$REACT_APP_MAPS_API_KEY" > .env.local
    !npm install
    !npm run build

print("\n✅ Build Complete.")
!cp public/index.html build/index.html
# ====================== PATCHING ======================
html_file_path = f"{target_dir}/build/index.html"
print(f"\n🔧 Patching {html_file_path}...")

if os.path.exists(html_file_path):
    with open(html_file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 1. Fix paths to be relative
    content = content.replace('src="/', 'src="./')
    content = content.replace('href="/', 'href="./')

    # 2. REMOVE the import.meta-breaking "type=module" hack
    content = content.replace('type="module" ', "")
    content = content.replace("type=module ", "")
    content = content.replace('<script type="module"', "<script")

    with open(html_file_path, "w", encoding="utf-8") as f:
        f.write(content)

    print("✅ Patched successfully — removed type=module + fixed paths")
else:
    print("❌ ERROR: build/index.html not found!")

# ====================== DEPLOY ======================
print("\n📦 Converting to UTF-16 and deploying...")
!cd {target_dir} && iconv -f UTF-8 -t UTF-16 ./build/*.html -o ./build/1ink.1ink
#!python3 deploy.py  # MAPS_API_KEY / REACT_APP_MAPS_API_KEY from Colab secrets
!python3 deploy.py  # MAPS_API_KEY / REACT_APP_MAPS_API_KEY from Colab secrets

print("\n🎉 Done! Check the live site.")

In [ ]:
# @title 🔨 The Jokesters
import os

# --- CONFIGURATION ---
PROJECT_NAME = "the_jokesters" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

!git clone https://huggingface.co/Supertone/supertonic /content/build_space/the_jokesters/public/tts

%cd {target_dir}

!git checkout main
#!git checkout hermes
#!git checkout imdrov-dev
#!git checkout improv-dev
#!git checkout fix-webllm-network-errors-7104480824362371282
#!git checkout improv2
#!git checkout copilot/add-multi-character-improv-function
#!git checkout copilot/improve-character-interaction
!git pull

# --- Conditional ONNX Model Download ---
onnx_models_dir = os.path.join(target_dir, "models", "onnx")
if not os.path.exists(onnx_models_dir):
    os.makedirs(onnx_models_dir)

# Map filenames to their download URLs
required_models = {
    "latent_denoiser.onnx": "https://1ink.us/files/onnx/latent_denoiser.onnx",
    "latent_denoiser.onnx_data": "https://1ink.us/files/onnx/latent_denoiser.onnx_data",
    "text_encoder.onnx": "https://1ink.us/files/onnx/text_encoder.onnx",
    "text_encoder.onnx_data": "https://1ink.us/files/onnx/text_encoder.onnx_data",
    "voice_decoder.onnx": "https://1ink.us/files/onnx/voice_decoder.onnx",
    "voice_decoder.onnx_data": "https://1ink.us/files/onnx/voice_decoder.onnx_data"
}

print(f"🔎 Checking models in {onnx_models_dir}...")

for filename, url in required_models.items():
    file_path = os.path.join(onnx_models_dir, filename)

    # Check if the specific file exists
    if not os.path.exists(file_path):
        print(f"⬇️ {filename} missing. Downloading...")
        # NOTE: -O (uppercase) is used to specify the output file.
        # -o (lowercase) would only write the log, not the data.
        !wget {url} -O {file_path}
    else:
        print(f"✅ {filename} found.")

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py
#!python3 deploy_models.py

In [ ]:
# @title 🔨 line_multiplication
import os

PROJECT_NAME = "line_multiplication"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Suno Bark Small
import os

PROJECT_NAME = "suno_bark_small_webgpu"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 FLAC Player (SDL3 Rebuild)
import os
import shutil

# --- CONFIGURATION ---
PROJECT_NAME = "flac_player"
USE_EMSDK = True

# Paths
target_dir = f"/content/build_space/{PROJECT_NAME}"
public_dir = os.path.join(target_dir, "public")
sdl_root_dir = os.path.join(target_dir, "src/sdl/build/SDL")
sdl_include_dir = os.path.join(sdl_root_dir, "include")
sdl_build_wasm_dir = os.path.join(sdl_root_dir, "build_wasm")
lib_dest_path = os.path.join(sdl_build_wasm_dir, "libSDL3.a")

# Source of pre-built files
lib_source_path = os.path.join(public_dir, "libSDL3.a")

# 1. Setup Environment
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"

if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository exists. Pulling latest...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
#!git checkout sdl3-audio
#!git checkout sdl3-audio
!git checkout main
#!git checkout sdl3-wasm-audio-heap-fix-13805891330175503067
!git pull

# 2. Smart Build Logic
print("🚀 Starting Build Setup...")

if USE_EMSDK:
    # --- STEP A: Ensure Headers Exist (Clone if needed) ---
    if not os.path.exists(sdl_include_dir):
        print("🔍 SDL3 headers not found. Cloning source code...")

        # If the directory exists but is empty/broken (no includes), clear it
        if os.path.exists(sdl_root_dir):
            print("   (Cleaning up incomplete SDL directory...)")
            shutil.rmtree(sdl_root_dir)

        # Clone SDL3 to get the headers
        !git clone https://github.com/libsdl-org/SDL.git {sdl_root_dir}
    else:
        print("✅ SDL3 source/headers already present.")

    # --- STEP B: Inject Pre-built Library ---
    if os.path.exists(lib_source_path):
        print(f"✅ Found pre-built libSDL3.a in {public_dir}")

        # Create the specific build_wasm folder if git clone didn't make it
        os.makedirs(sdl_build_wasm_dir, exist_ok=True)

        # Copy the lib. This stops build.sh from running the slow compilation.
        shutil.copy2(lib_source_path, lib_dest_path)
        print(f"📋 Injected libSDL3.a into {sdl_build_wasm_dir}")
    else:
        print("⚠️ libSDL3.a not found in public/. Build script will likely recompile it.")

    # --- STEP C: Run the Build ---
    print("🔨 Running Audio Engine Build...")
    !chmod +x {target_dir}/src/sdl/build.sh
    !bash {target_dir}/src/sdl/build.sh

    # --- STEP D: NPM Build ---
    print("📦 Running NPM Build...")
    !npm install
    !npm run build

else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")


import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py

In [ ]:
# @title 🔨 Rain
import os

PROJECT_NAME = "rain"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 Platformer
import os

# --- CONFIGURATION ---
PROJECT_NAME = "wasm-platformer" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
#!git checkout fix-hover-and-expand-map
#!git checkout dynamic-sprite-offset
!git checkout fix-run-hovering
!git pull


# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    !emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="dist/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 vanilla-arcade
import os

PROJECT_NAME = "vanilla-arcade"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 webgpu-ts
import os

PROJECT_NAME = "webgpu-ts"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "fix-webgpu-texture-mismatch"
HTML_SUBPATH = "build/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 rain_chime
import os

PROJECT_NAME = "rain_chime"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 os13k web_mouse
import os

#**How to run:**
#1. `yarn install`
#2. `yarn start`
#3. Open `http://localhost:8080` (OS13k Receiver).
#4. Open `http://localhost:8080/controller` (Phone Controller) on a separate device or tab.
#5. Enter the ID displayed on the OS13k screen into the controller to connect.

# --- CONFIGURATION ---
PROJECT_NAME = "os13k" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout web_mouse

!git pull


# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !yarn install
    !chmod +x build.sh
    !./build.sh

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="dist/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 supertonic-TTS
import os

# --- CONFIGURATION ---
PROJECT_NAME = "supertonic-tts" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"
!git clone https://huggingface.co/Supertone/supertonic /content/build_space/supertonic-tts/assets

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}/web
!git pull

!git checkout sing
!git pull

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build


print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}/web"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
!cd /content/build_space/{PROJECT_NAME}/web && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink

!python3 deploy.py

In [ ]:
# @title 🔨 Lite Brite
import os

PROJECT_NAME = "litebrite-webgpu"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 dog_man
import os

PROJECT_NAME = "dog_man"
GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git"
USE_EMSDK = False
BRANCH = "main"
HTML_SUBPATH = "dist/index.html"
DEPLOY_SCRIPT = "deploy.py"
RECURSIVE_CLONE = False

target_dir = clone_or_pull(PROJECT_NAME, branch=BRANCH, recursive=RECURSIVE_CLONE, git_repo=GIT_REPO)
print("🚀 Starting Build...")
run_npm_build(PROJECT_NAME, use_emsdk=USE_EMSDK, cwd=target_dir)
print("✅ Build Complete.")
finalize_deploy(PROJECT_NAME, html_subpath=HTML_SUBPATH, deploy_script=DEPLOY_SCRIPT)

In [ ]:
# @title 🔨 IPTV Player
import os

# --- CONFIGURATION ---
PROJECT_NAME = "iptv" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout custom

!git pull
!npm install
%cd {target_dir}/web

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Standard NPM Build
    !npm install

    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.
!mkdir /content/build_space/{PROJECT_NAME}/web/dist/streams
#!cp /content/build_space/{PROJECT_NAME}/streams/* /content/build_space/{PROJECT_NAME}/web/dist/streams/

target_dir = f"/content/build_space/{PROJECT_NAME}/web"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

# PROJECT_NAME is defined in a previous cell and inherited here.
!cd /content/build_space/{PROJECT_NAME}/web && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py

# Build Other 🔨

In [ ]:
# @title 🔨 libmodplug #1
import os

# --- CONFIGURATION ---
PROJECT_NAME = "openmpt" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

#!git checkout main
!git checkout js_interface
!git pull
!mkdir /content/build_space/jc303_wasm/build && cd /content/build_space/jc303_wasm/build

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")
!source /content/build_space/emsdk/emsdk_env.sh && cd /content/build_space/jc303_wasm/build && emmake make clean
!source /content/build_space/emsdk/emsdk_env.sh && cd /content/build_space/jc303_wasm/build && emmake make -j16 VERBOSE=1 CONFIG=emscripten EMSCRIPTEN_TARGET=1it1-new3 CXXFLAGS="-DMPT_ENABLE_SAVECREATE_XM -DMPT_ENABLE_SAVING" NO_SNDFILE=0

print("✅ Build Complete.")


In [ ]:
# @title 🔨 libmodplug #2
%%shell
export JVM_HEAP_SIZE=8g
cd /content/build_space/openmpt/bin
google-closure-compiler --js libopenmpt.js --chunk_output_type=ES_MODULES --js_output_file libopenmptg.js --language_in UNSTABLE --language_out UNSTABLE --compilation_level SIMPLE_OPTIMIZATIONS
#google-closure-compiler --js libopenmpt.js --js_output_file libopenmptg.js --language_in ECMASCRIPT_2021 --compilation_level SIMPLE_OPTIMIZATIONS
# terser -o ./libopenmptt.js --compress --comments false --mangle -- libopenmptg.js
#terser -o ./libopenmptt.js -c -m -comments false -- libopenmptg.js
#terser -o ./libopenmptt.js -c -- libopenmptg.js
 #  wasm-opt libopenmpt.wasm -O0 -all -o libopenmpt.wasm
iconv -f UTF-8 -t UTF16LE /content/build_space/openmpt/bin/libopenmpt.js -o /content/RAMDRIVE2/openmpt/bin/libopenmptjs.1ijs
iconv -f UTF-8 -t UTF-32 /content/build_space/openmpt/bin/libopenmpt.js -o /content/RAMDRIVE2/openmpt/bin/libopenmptjs.3ijs


In [ ]:
# @title 🔨 libmodplug #3
loc_file = "libopenmptjs.1ijs" #@param ["sh4.1ijs", "sh5.1ijs", "g3007.wasm", "g3008.wasm", "g3009.wasm", "sh6.1ijs", "g3010.wasm"] {allow-input: true}
loc_fileb = "libopenmpt.js" #@param ["sh4.1ijs", "sh5.1ijs", "g3007.wasm", "g3008.wasm", "g3009.wasm", "sh6.1ijs", "g3010.wasm"] {allow-input: true}
loc_file4 = "libopenmptjs.3ijs" #@param ["sh4.1ijs", "sh5.1ijs", "g3007.wasm", "g3008.wasm", "g3009.wasm", "sh6.1ijs", "g3010.wasm"] {allow-input: true}
import os
import urllib
import requests as reqs
import re
import paramiko
host = "1ink.us"
username = os.environ.get("DEPLOY_USER", os.environ.get("SFTP_USER", ""))
password = os.environ.get("DEPLOY_PASS", os.environ.get("SFTP_PASSWORD", ""))
port = 22
file_name=loc_fileb
local_path="/content/build_space/openmpt/bin/"+file_name
transport = paramiko.Transport((host, port))
destination_path="wasm.noahcohn.com/libmpt2/"+file_name
transport.connect(username = username, password = password)
sftp = paramiko.SFTPClient.from_transport(transport)
sftp.put(local_path, destination_path)
sftp.close()
transport.close()
file_name=loc_file
local_path="/content/build_space/openmpt/bin/"+file_name
transport = paramiko.Transport((host, port))
destination_path="wasm.noahcohn.com/libmpt2/"+file_name
transport.connect(username = username, password = password)
sftp = paramiko.SFTPClient.from_transport(transport)
sftp.put(local_path, destination_path)
sftp.close()
transport.close()
loc_file2 = "libopenmpt.wasm" #@param ["sh4.1ijs", "sh5.1ijs", "g3007.wasm", "g3008.wasm", "g3009.wasm", "sh6.1ijs", "g3010.wasm"] {allow-input: true}
import os
import urllib
import requests as reqs
import re
import paramiko
host = "1ink.us"
username = os.environ.get("DEPLOY_USER", os.environ.get("SFTP_USER", ""))
password = os.environ.get("DEPLOY_PASS", os.environ.get("SFTP_PASSWORD", ""))
port = 22
file_name=loc_fileb
local_path="/content/build_space/openmpt/bin/"+file_name
transport = paramiko.Transport((host, port))
destination_path="wasm.noahcohn.com/libmpt2/libopenmptjs.js"
transport.connect(username = username, password = password)
sftp = paramiko.SFTPClient.from_transport(transport)
#sftp.put(local_path, destination_path)
sftp.close()
transport.close()
loc_file3 = "libopenmpt.js.mem" #@param ["sh4.1ijs", "sh5.1ijs", "g3007.wasm", "g3008.wasm", "g3009.wasm", "sh6.1ijs", "g3010.wasm"] {allow-input: true}
import os
import urllib
import requests as reqs
import re
import paramiko
host = "1ink.us"
username = os.environ.get("DEPLOY_USER", os.environ.get("SFTP_USER", ""))
password = os.environ.get("DEPLOY_PASS", os.environ.get("SFTP_PASSWORD", ""))
port = 22
file_name=loc_file3
local_path="/content/build_space/openmpt/bin/"+file_name
transport = paramiko.Transport((host, port))
destination_path="wasm.noahcohn.com/libmpt2/"+file_name
transport.connect(username = username, password = password)
sftp = paramiko.SFTPClient.from_transport(transport)
#sftp.put(local_path, destination_path)
sftp.close()
transport.close()

In [ ]:
# @title 🔨 JC-303 WASM
import os

# --- CONFIGURATION ---
PROJECT_NAME = "jc303_wasm" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = True # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}
!git checkout main
!git pull
!mkdir /content/build_space/jc303_wasm/build && cd /content/build_space/jc303_wasm/build

# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && cd /content/build_space/jc303_wasm/build && emcmake cmake ..
    !source /content/build_space/emsdk/emsdk_env.sh && cd /content/build_space/jc303_wasm/build && emmake make -j55
else:
    # Standard NPM Build
    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="dist/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 AnuraOS
import os

print("⬇️ Installing Anura Dependencies...")

# 1. Check & Install Rust if missing
if not os.path.exists(os.path.expanduser("~/.cargo/env")):
    print("⚠️ Rust not found. Installing Rust...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# 2. Install system dependencies
# NOTE: Removing 'cargo' from apt to avoid conflict with rustup
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    openjdk-11-jdk \
    inotify-tools \
    gcc-multilib \
    libc6-dev-i386 \
    clang \
    uuid-runtime \
    jq \
    docker.io \
    binaryen

# 3. Ensure Rust targets are available
!source "$HOME/.cargo/env" && rustup target add wasm32-unknown-unknown && rustup target add i686-unknown-linux-gnu

# --- CONFIGURATION ---
PROJECT_NAME = "anuraOS" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir} --recursive
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git reset --hard
    !git pull

%cd {target_dir}

import os

# --- CONFIGURATION ---
TARGET_DIR = "/content/build_space/anuraOS"
DEPLOY_PREFIX = "/anura"
DOUBLE_PREFIX = "/anura/anura"

print("🚑 STARTING EMERGENCY PATH CORRECTION...")

def fix_double_paths(file_path):
    if not os.path.exists(file_path):
        return

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    original_content = content

    # FIX 1: Replace double "/anura/anura/" with "/anura/"
    if DOUBLE_PREFIX in content:
        content = content.replace(DOUBLE_PREFIX, DEPLOY_PREFIX)

    # FIX 2: Check for potential triple prefixes just in case
    triple_prefix = "/anura/anura/anura"
    if triple_prefix in content:
        content = content.replace(triple_prefix, DEPLOY_PREFIX)

    if content != original_content:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
        print(f"  ✅ Fixed double-patch in: {os.path.basename(file_path)}")
    else:
        print(f"  ℹ️  No double-patch found in: {os.path.basename(file_path)}")

# List of files we know were affected
files_to_check = [
    f"{TARGET_DIR}/src/Boot.tsx",
    f"{TARGET_DIR}/src/Bootsplash.tsx",
    f"{TARGET_DIR}/src/api/Filesystem.ts",
    f"{TARGET_DIR}/src/networking/Networking.ts",
    f"{TARGET_DIR}/public/anura-sw.js",
    f"{TARGET_DIR}/public/index.html",
    f"{TARGET_DIR}/config.default.json"
]

# Run the fix
for file_path in files_to_check:
    fix_double_paths(file_path)

# Also fix CSS files
print("  🔧 Scanning CSS files for double paths...")
os.system(f"grep -rl '{DOUBLE_PREFIX}' {TARGET_DIR}/src | xargs sed -i 's|{DOUBLE_PREFIX}|{DEPLOY_PREFIX}|g'")

print("🏁 CORRECTION COMPLETE.")
# --- FIX: RELAX ESLINT CONFIG ---
eslint_path = f"{target_dir}/eslint.config.mjs"
if os.path.exists(eslint_path):
    print("🔧 Patching eslint.config.mjs to ignore undefined variables...")
    with open(eslint_path, "r") as f:
        content = f.read()
    # Add "no-undef": "off" to rules
    if '"@typescript-eslint/no-unused-expressions": "off",' in content:
        content = content.replace(
            '"@typescript-eslint/no-unused-expressions": "off",',
            '"@typescript-eslint/no-unused-expressions": "off",\n\t\t\t"no-undef": "off",'
        )
    with open(eslint_path, "w") as f:
        f.write(content)


# --- BUILD COMMANDS ---
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
else:
    # Standard NPM Build
    !source "$HOME/.cargo/env" && make static -j55
    !npm install

print("✅ Build Complete.")

# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
# Path to static/index.html for final check
html_file_path = f'{target_dir}/static/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'make static' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")

!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!python3 deploy.py

In [ ]:
# @title 🔨 AnuraOS (build 2)
import os

# --- CONFIGURATION ---
PROJECT_NAME = "anuraOS"
TARGET_DIR = f"/content/build_space/{PROJECT_NAME}"

print("🚀 Starting Recovery Build...")

# 1. ENSURE REPO & SUBMODULES
if not os.path.exists(TARGET_DIR):
    # Clone if missing
    !git clone --recursive https://github.com/ford442/anuraOS-agent.git {TARGET_DIR}
    %cd {TARGET_DIR}
    !git checkout subdirectory-support-wisp-config
else:
    # Reset if existing
    %cd {TARGET_DIR}
    print("🔄 resetting repository state...")
    !git reset --hard
    !git pull

print("📦 Updating submodules (v86)...")
!git submodule update --init --recursive

# 2. PATCH MAKEFILE (Fix the Hang)
makefile_path = "Makefile"
if os.path.exists(makefile_path):
    print("🔧 Patching Makefile to prevent 'cp' hang...")
    with open(makefile_path, "r") as f:
        content = f.read()

    # 1. Fix the 'cp -r' hang by forcing overwrite (-f) and removing destination first
    # We replace "cp -r v86/bios public" with a safer sequence
    if "cp -r v86/bios public" in content:
        content = content.replace(
            "cp -r v86/bios public",
            "rm -rf public/bios && mkdir -p public/bios && cp -rf v86/bios/* public/bios/"
        )
        print("  ✅ Fixed v86/bios copy command.")

    # 2. Fix the 'lint' dependency if strictly necessary (skip linting for speed)
    content = content.replace("bundle: tsc css lint milestone", "bundle: tsc css milestone")

    with open(makefile_path, "w") as f:
        f.write(content)

# 3. VERIFY V86 BIOS
bios_path = "v86/bios"
if not os.path.exists(bios_path) or not os.listdir(bios_path):
    print("⚠️ Warning: v86/bios seems empty. Attempting to fetch manually...")
    # Fallback: Download generic seabios/vgabios if missing (Anura usually needs these)
    !mkdir -p v86/bios
    !wget -q -O v86/bios/seabios.bin https://github.com/copy/v86/raw/master/bios/seabios.bin
    !wget -q -O v86/bios/vgabios.bin https://github.com/copy/v86/raw/master/bios/vgabios.bin
    print("  ✅ Downloaded fallback BIOS files.")

# 4. CLEAN & BUILD
print("🔨 Building...")
# Clean old artifacts to prevent caching issues
!rm -rf build/
!make clean

# Run build
# We skip the 'npm install' inside the makefile if it hangs, running it manually first often helps
!npm install
!source "$HOME/.cargo/env" && make static -j$(nproc)

# 5. DEPLOY
print("🚀 Deploying...")
if os.path.exists("deploy.py"):
    !python3 deploy.py
else:
    print("❌ deploy.py not found.")

print("✅ Process Complete.")

In [ ]:
# @title 🔨 visual6502
import os

# --- CONFIGURATION ---
PROJECT_NAME = "visual6502" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL
USE_EMSDK = False # Set to True if you need C++
# ---------------------

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git checkout master

!git pull


# --- BUILD COMMANDS ---
# Modify the section below based on your project needs
print("🚀 Starting Build...")

if USE_EMSDK:
    # Example C++ Build
    !source /content/build_space/emsdk/emsdk_env.sh && npm install && npm run build
    #!emcc /content/build_space/wasm-platformer/cpp/src/main.cpp /content/build_space/wasm-platformer/cpp/src/Game.cpp -o /content/build_space/wasm-platformer/public/game.js -O3 -s WASM=1 -s MODULARIZE=1 -s 'EXPORT_NAME="createGameModule"' -lembind

    !npm install
    !npm run build

else:
    # Standard NPM Build

    !npm install
    !npm run build

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")


# PROJECT_NAME is defined in a previous cell and inherited here.
if os.path.exists(html_file_path):
    finalize_deploy(PROJECT_NAME, html_subpath="dist/index.html", deploy_script="deploy.py", skip_patch=True)
else:
    print(f"❌ Skipping deploy — {html_file_path} not found")


In [ ]:
# @title 🔨 sinsy

import os

# --- CONFIGURATION ---
PROJECT_NAME = "sinsy" # @param ["mod-player", "your-other-project-name"] # Customize with your project names

GIT_REPO = f"https://github.com/ford442/{PROJECT_NAME}.git" # <--- CHANGE THIS TO YOUR REPO URL

target_dir = f"/content/build_space/{PROJECT_NAME}"

# Clone if not exists, otherwise pull latest changes
if not os.path.exists(target_dir):
    print(f"⬇️ Cloning {GIT_REPO}...")
    !git clone {GIT_REPO} {target_dir}
else:
    print(f"🔄 Repository {PROJECT_NAME} already exists. Pulling latest changes...")
    %cd {target_dir}
    !git pull

%cd {target_dir}

!git pull
!git checkout copilot/fix-garbled-audio-output
!bash ./build.sh

print("✅ Build Complete.")

import os

# --- CONFIGURATION ---
# PROJECT_NAME is defined in a previous cell and inherited here.

target_dir = f"/content/build_space/{PROJECT_NAME}"
html_file_path = f'{target_dir}/dist/index.html'
print(f"Attempting to patch {html_file_path}...")
if not os.path.exists(html_file_path):
    print(f"❌ ERROR: File not found at {html_file_path}. Did 'npm run build' complete successfully?")
else:
    try:
        with open(html_file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        content = content.replace('src="/', 'src="./')
        content = content.replace('href="/', 'href="./')
        content = content.replace("src='/", "src='./")
        content = content.replace("href='/", "href='./")
        with open(html_file_path, 'w', encoding='utf-8') as file:
            file.write(content)
        print("✅ Successfully updated paths in index.html to be relative.")
    except Exception as e:
        print(f"❌ An error occurred during patching: {e}")
!cd /content/build_space/{PROJECT_NAME} && iconv -f UTF-8 -t UTF-16 ./dist/index.html -o ./dist/1ink.1ink
!mkdir -p ./dist
!cp ./index.html ./dist/index.html
!cp ./bin/sinsy.js ./dist/sinsy.js
!cp ./jszip.min.js ./dist/jszip.min.js
!cp ./bin/sinsy.data ./dist/sinsy.data
!cp ./bin/sinsy.wasm ./dist/sinsy.wasm
# PROJECT_NAME is defined in a previous cell and inherited here.
!python3 deploy.py

In [ ]:
#@title Rebuild SDL3

import os
import shutil

PROJECT_NAME = "flac_player"
target_dir = f"/content/build_space/{PROJECT_NAME}"

# 1. CLEAN: Remove incompatible library and old outputs
sdl_wasm_build = os.path.join(target_dir, "src/sdl/build/SDL/build_wasm")
public_lib = os.path.join(target_dir, "public/libSDL3.a")
public_js = os.path.join(target_dir, "public/sdl-audio.js")
public_wasm = os.path.join(target_dir, "public/sdl-audio.wasm")

print("🧹 Cleaning old build artifacts...")
if os.path.exists(sdl_wasm_build):
    shutil.rmtree(sdl_wasm_build)
if os.path.exists(public_lib):
    os.remove(public_lib)
if os.path.exists(public_js):
    os.remove(public_js)
if os.path.exists(public_wasm):
    os.remove(public_wasm)

# 2. BUILD: Run the updated build script
print("🔨 Building with new flags (WASM_WORKERS + HEAPF32 export)...")
!chmod +x {target_dir}/src/sdl/build.sh
!bash {target_dir}/src/sdl/build.sh

# 3. NPM Build
print("📦 Packaging...")
!cd {target_dir} && npm run build

# MAME GnG

In [ ]:
%%shell
cd /content/build_space
git clone https://github.com/ford442/mame.git -b vite2
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
make clean

In [ ]:
%%shell
cd /content/build_space
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
# --- STEP 1: GENERATE PROJECT FILES ---
# We use standard 'make' (not emmake) and NO web flags.
# This builds the 'genie' tool using GCC without crashing.
make -j55 \
 SUBTARGET=gng_web \
 OSD=sdl \
 REGENIE=1 \
 TOOLS=0 \
 USE_QTDEBUG=0 \
 NOWERROR=1 \
 USE_BGFX=0 \
 SOURCES=src/mame/capcom/gng.cpp \
 generate


In [ ]:
%%shell
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
git pull
# --- STEP 2: COMPILE THE GAME ---
# Now we use 'emmake' and pass the web flags.
# We set REGENIE=0 so it doesn't try to use GCC again.

emmake make \
 SUBTARGET=gng_web \
 OSD=sdl \
 REGENIE=0 \
 TOOLS=0 \
 USE_QTDEBUG=0 \
 NOWERROR=1 \
 USE_BGFX=0 \
 SOURCES=src/mame/capcom/gng.cpp \
 LDFLAGS="-lGL -sASYNCIFY -sALLOW_MEMORY_GROWTH=1 -sDISABLE_EXCEPTION_CATCHING=0 -sLEGACY_GL_EMULATION=1 -sFORCE_FILESYSTEM=1 -sUSE_SDL=2 -sASSERTIONS=0 --embed-file /content/build_space/mame/roms/gng.zip@/roms/gng.zip" \
 -j55

In [ ]:
!cd /content/build_space/mame/web && npm i
!cd /content/build_space/mame/web && npm run build

In [ ]:
!cd /content/build_space/mame/ && python3 deploy.py

In [ ]:
%%shell
cd /content/build_space
git clone https://github.com/ford442/mame.git -b vite2

#git clone https://github.com/db48x/emularity.git

cd /content/build_space/mame
git pull
source /content/build_space/emsdk/emsdk_env.sh

### make clean

#emmake make SUBTARGET=gng_web REGENIE=1 TOOLS=0 VIDEO=bgfx SOUND=js TARGET_LIBS="-lGL -sFORCE_FILESYSTEM=1 -sFULL_ES2=1 -sFULL_ES3=1 -sUSE_SDL=2 -sASSERTIONS=2  --preload-file ./roms/gng.zip@/roms/gng.zip" SOURCES=src/mame/capcom/gng.cpp -j55
emmake make SUBTARGET=gng_web OSD=sdl REGENIE=1 TOOLS=0 VIDEO=bgfx SOUND=js LDFLAGS="-lGL -sASYNCIFY=1 -sALLOW_MEMORY_GROWTH=1 -sAUDIO_WORKLET=1 -sLEGACY_GL_EMULATION=1 -sFORCE_FILESYSTEM=1 -sUSE_SDL=2 -sASSERTIONS=1 --preload-file /content/build_space/mame/roms/gng.zip@/roms/gng.zip" SOURCES=src/mame/capcom/gng.cpp -j55

#Gemini3.0 new way -->
'''
emmake make \
 SUBTARGET=gng_web \
 OSD=sdl \
 REGENIE=0 \
 TOOLS=0 \
 USE_QTDEBUG=0 \
 NOWERROR=1 \
 USE_BGFX=0 \
 SOURCES=src/mame/capcom/gng.cpp \
 LDFLAGS="-lGL -sASYNCIFY -sALLOW_MEMORY_GROWTH=1 -sLEGACY_GL_EMULATION=1 -sFORCE_FILESYSTEM=1 -sUSE_SDL=2 -sASSERTIONS=0 --embed-file /content/build_space/mame/roms/gng.zip@/roms/gng.zip" \
 -j55
 '''

# MAME Snow Bros

In [ ]:
%%shell
cd /content/build_space
git clone https://github.com/ford442/mame.git -b vite2
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
make clean

In [ ]:
%%shell
cd /content/build_space
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
# --- STEP 1: GENERATE PROJECT FILES ---
# We use standard 'make' (not emmake) and NO web flags.
# This builds the 'genie' tool using GCC without crashing.
make -j55 \
 SUBTARGET=snowbros \
 OSD=sdl \
 REGENIE=1 \
 TOOLS=0 \
 USE_QTDEBUG=0 \
 NOWERROR=1 \
 USE_BGFX=0 \
 SOURCES=src/mame/capcom/snowbros.cpp \
 generate


In [ ]:
%%shell
source /content/build_space/emsdk/emsdk_env.sh
cd /content/build_space/mame
git pull
# --- STEP 2: COMPILE THE GAME ---
# Now we use 'emmake' and pass the web flags.
# We set REGENIE=0 so it doesn't try to use GCC again.

emmake make \
 SUBTARGET=snowbros \
 OSD=sdl \
 REGENIE=0 \
 TOOLS=0 \
 USE_QTDEBUG=0 \
 NOWERROR=1 \
 USE_BGFX=0 \
 SOURCES=toaplan/toaplan1.cpp \
 LDFLAGS="-lGL -sASYNCIFY -sALLOW_MEMORY_GROWTH=1 -sDISABLE_EXCEPTION_CATCHING=0 -sLEGACY_GL_EMULATION=1 -sFORCE_FILESYSTEM=1 -sUSE_SDL=2 -sASSERTIONS=0 --embed-file /content/build_space/mame/roms/snowbros.zip@/roms/snowbros.zip" \
 -j55

In [ ]:
!cd /content/build_space/mame/web && npm i
!cd /content/build_space/mame/web && npm run build-snowbros

In [ ]:
!cd /content/build_space/mame/ && python3 deploy_snowbros.py